In [2]:
!pip install -U datasets transformers evaluate accelerate tdqm nltk sentencepiece rouge_score gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

We almost mostly use pretrained models and fine-tuning them for new use cases by reusing the weights from pretraining. This is commonly referred to as transfer learning, and it’s a very successful strategy for applying Transformer models to most real-world use cases where labeled data is sparse.

Now, lets take a different approach and train a completely new model from scratch. This is a good approach to take if we have a lot of data and it is very different from the pretraining data used for the available models. However, it also requires considerably more compute resources to pretrain a language model than just to fine-tune an existing one.

Examples where it can make sense to train a new model include for datasets consisting of musical notes, molecular sequences such as DNA, or programming languages. The latter have recently gained traction thanks to tools such as TabNine and GitHub’s Copilot, powered by OpenAI’s Codex model, that can generate long sequences of code. This task of text generation is best addressed with auto-regressive or causal language models such as GPT-2.

In this notebook, lets build a scaled-down version of a code generation model, focusing on one-line completions instead of full functions or classes, using a subset of Python code. When working with data in Python you are in frequent contact with the Python data science stack, consisting of the matplotlib, seaborn, pandas, and scikit-learn libraries. When using those frameworks it’s common to need to look up specific commands, so it would be nice if we could use a model to complete these calls for us.

### 1. Load the data
Python code is abundantly available from code repositories such as GitHub, which we can use to create a dataset by scraping for every Python repository. This was the approach taken in the Transformers textbook to pretrain a large GPT-2 model. Using a GitHub dump of about 180 GB containing roughly 20 million Python files called `codeparrot`, the authors built a dataset that they then shared on the Hugging Face Hub.

#### The Codeparrot dataset - streaming dataset

However, training on the full corpus is time- and compute-consuming, and we only need the subset of the dataset concerned with the Python data science stack. So, let’s start by filtering the codeparrot dataset for all files that include any of the libraries in this stack. Because of the dataset’s size, we want to avoid downloading it; instead, we’ll use the streaming feature to filter it on the fly.

In [12]:
def check_for_keywords(string, keywords):
    """
    Checks if the string contains any of the keywords. Returns True if yes, else False
    """
    return any(kw.lower() in string.lower() for kw in keywords)

In [13]:
# Test
filters = ["pandas", "numpy", "matplotlib", "torch", "transformers", "seaborn", "sklearn"]
code_1 = "import numpy as np"
code_2 = "from transformers import AutoModel"
code_3 = "import tensorflow as tf"

print(check_for_keywords(code_1, filters))
print(check_for_keywords(code_2, filters))
print(check_for_keywords(code_3, filters))

True
True
False


Lets create a function that will stream the dataset and filter the elements we need

In [14]:
from collections import defaultdict
from tqdm import tqdm
from datasets import Dataset


def filter_streaming_dataset(dataset, filters):
    filtered_dict = defaultdict(list)
    total = 0
    for sample in tqdm(iter(dataset)):
        if check_for_keywords(sample["content"], filters):
            for k, v in sample.items():
                filtered_dict[k].append(v)
        total += 1
    filtered_dataset = Dataset.from_dict(filtered_dict)
    print(f"{len(filtered_dict['content']) / total:.2%} of data after filtering.")
    return filtered_dataset

In [15]:
from datasets import load_dataset

# Load the train split in streaming mode
split="train" # "valid"
ds = load_dataset(f"transformersbook/codeparrot-{split}", split=split, streaming=True)  # split='train[:1%]' dont work for streaming ds

README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

In [16]:
ds

IterableDataset({
    features: Unknown,
    num_shards: 183
})

In [17]:
# len(iter(list(ds))) # dont run...data is huge

In [18]:
for sample in iter(ds):
    break
sample

{'repo_name': 'ahmedbodi/AutobahnPython',
 'path': 'examples/asyncio/websocket/echo/client_coroutines.py',
 'copies': '13',
 'size': '2044',
 'content': '###############################################################################\n##\n##  Copyright (C) 2013-2014 Tavendo GmbH\n##\n##  Licensed under the Apache License, Version 2.0 (the "License");\n##  you may not use this file except in compliance with the License.\n##  You may obtain a copy of the License at\n##\n##      http://www.apache.org/licenses/LICENSE-2.0\n##\n##  Unless required by applicable law or agreed to in writing, software\n##  distributed under the License is distributed on an "AS IS" BASIS,\n##  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n##  See the License for the specific language governing permissions and\n##  limitations under the License.\n##\n###############################################################################\n\nfrom autobahn.asyncio.websocket import WebSocketClien

The content field contains the code that we want our model to train on.

In [19]:
sample["content"]

'###############################################################################\n##\n##  Copyright (C) 2013-2014 Tavendo GmbH\n##\n##  Licensed under the Apache License, Version 2.0 (the "License");\n##  you may not use this file except in compliance with the License.\n##  You may obtain a copy of the License at\n##\n##      http://www.apache.org/licenses/LICENSE-2.0\n##\n##  Unless required by applicable law or agreed to in writing, software\n##  distributed under the License is distributed on an "AS IS" BASIS,\n##  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n##  See the License for the specific language governing permissions and\n##  limitations under the License.\n##\n###############################################################################\n\nfrom autobahn.asyncio.websocket import WebSocketClientProtocol, \\\n                                       WebSocketClientFactory\n\nimport asyncio\n\n\n\nclass MyClientProtocol(WebSocketClientProtocol):\n\

In [20]:
# Apply filter to streaming dataset
# filters = ["pandas", "numpy", "matplotlib", "torch", "transformers", "seaborn", "sklearn"]
# filtered_data = filter_streaming_dataset(ds, filters)

This leaves us with about 3% of the original train dataset, which is still quite sizable — the resulting dataset is 6 GB and consists of 600,000 Python scripts!

Filtering the full dataset can take 2-3h depending on colab GPU and bandwidth. If we don’t want to go through this lengthy process yourself, Hugging face provide the filtered dataset on the Hub for us to download.

In [21]:
from datasets import load_dataset, DatasetDict

train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
validation = load_dataset("huggingface-course/codeparrot-ds-valid", split="validation")

raw_ds = DatasetDict({
    "train": train.shuffle(seed=42).select(range(50000)),  # 606720
    "validation": validation.shuffle(seed=42).select(range(500))  # 3322
})

codeparrot-ds-train.jsonl:   0%|          | 0.00/8.25G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

codeparrot-ds-valid.jsonl:   0%|          | 0.00/46.1M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/3322 [00:00<?, ? examples/s]

In [22]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 500
    })
})

In [23]:
def show_samples(ds, num_samples=1):
    for sample in range(num_samples):
        for col_name in ds.column_names:
            print(f"{col_name.upper()}: {ds[col_name][sample][:200]}")

In [24]:
show_samples(raw_ds["train"])

REPO_NAME: ThomasMiconi/htmresearch
PATH: projects/feedback/feedback_sequences.py
COPIES: 2
SIZE: 26875
CONTENT: 
# Numenta Platform for Intelligent Computing (NuPIC)
# Copyright (C) 2016, Numenta, Inc.  Unless you have an agreement
# with Numenta, Inc., for a separate license for this software code, the
# follo
LICENSE: agpl-3.0


Now that we have a dataset, we need to preprocess the texts so they’re in a format suitable for pretraining.

### 2. Preprocessing

The first step will be to tokenize the data, so we can use it for training. Since our goal is to mainly autocomplete short function calls, we can keep the context size relatively small. This has the benefit that we can train the model much faster and it requires significantly less memory. If it is important for our application to have more context (for example, if we want the model to write unit tests based on a file with the function definition), we have to make sure to increase context length number, but also keep in mind that this comes with a greater GPU memory footprint. For now, let’s fix the context size at 128 tokens, as opposed to the 1,024 or 2,048 used in GPT-2 or GPT-3, respectively.

Most documents contain many more than 128 tokens, so simply truncating the inputs to the maximum length would eliminate a large fraction of our dataset. Instead, we’ll use the `return_overflowing_tokens` option to tokenize the whole input and split it into several chunks. We’ll also use the `return_length` option to return the length of each created chunk automatically. Often the last chunk will be smaller than the context size, and we’ll get rid of these pieces to avoid padding issues; we don’t really need them as we have plenty of data anyway.

In [25]:
from transformers import AutoTokenizer

checkpoint = "huggingface-course/code-search-net-tokenizer"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

GPT2TokenizerFast(name_or_path='huggingface-course/code-search-net-tokenizer', vocab_size=50000, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [26]:
max_input_seq_length = 128 # context length

def preprocess(examples):
    inputs = tokenizer(examples["content"], max_length=max_input_seq_length, truncation=True, return_overflowing_tokens=True, return_length=True)
    input_batch = []
    for input_ids, length in zip(inputs["input_ids"], inputs["length"]):
        if length == max_input_seq_length: # dropout the small input_ids < max_input_seq_length
            input_batch.append(input_ids)
    return {"input_ids": input_batch}

In [27]:
# Test
inputs = tokenizer(raw_ds["train"][0]["content"], max_length=max_input_seq_length, truncation=True, return_overflowing_tokens=True, return_length=True)

print(f"Inputs: {dict(inputs).keys()}")
print(f'Original Sample: {len(raw_ds["train"][0]["content"].split())}')
print(f"Num of Input chunks: {len(inputs['input_ids'])}") # 59 chunka
print(f"Input chunk lengths: {(inputs['length'])}")
print(f"Chunk mapping to original sample: {inputs['overflow_to_sample_mapping']}")

Inputs: dict_keys(['input_ids', 'attention_mask', 'length', 'overflow_to_sample_mapping'])
Original Sample: 2252
Num of Input chunks: 59
Input chunk lengths: [128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 17]
Chunk mapping to original sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Observe that the last chunk in the inputs for sample 0 is 17 whilst others have seq length of 128. So, we need padding to avoid breakage.

In [28]:
tokenized_ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds["train"].column_names)
tokenized_ds

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 1375550
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 13617
    })
})

In [35]:
raw_ds["train"][0]["content"]

'\n# Numenta Platform for Intelligent Computing (NuPIC)\n# Copyright (C) 2016, Numenta, Inc.  Unless you have an agreement\n# with Numenta, Inc., for a separate license for this software code, the\n# following terms and conditions apply:\n#\n# This program is free software: you can redistribute it and/or modify\n# it under the terms of the GNU Affero Public License version 3 as\n# published by the Free Software Foundation.\n#\n# This program is distributed in the hope that it will be useful,\n# but WITHOUT ANY WARRANTY; without even the implied warranty of\n# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.\n# See the GNU Affero Public License for more details.\n#\n# You should have received a copy of the GNU Affero Public License\n# along with this program.  If not, see http://www.gnu.org/licenses.\n#\n# http://numenta.org/licenses/\n# ----------------------------------------------------------------------\n\n"""\nThis file runs a number of experiments testing the effectiveness of 

In [34]:
tokenizer.convert_ids_to_tokens(tokenized_ds["train"][0]["input_ids"])

['Ċ',
 '#',
 'ĠN',
 'ument',
 'a',
 'ĠPlatform',
 'Ġfor',
 'ĠIntel',
 'lig',
 'ent',
 'ĠComputing',
 'Ġ(',
 'Nu',
 'PIC',
 ')',
 'Ċ',
 '#',
 'ĠCopyright',
 'Ġ(',
 'C',
 ')',
 'Ġ2016',
 ',',
 'ĠN',
 'ument',
 'a',
 ',',
 'ĠInc',
 '.',
 'Ġ',
 'ĠUnless',
 'Ġyou',
 'Ġhave',
 'Ġan',
 'Ġagreement',
 'Ċ',
 '#',
 'Ġwith',
 'ĠN',
 'ument',
 'a',
 ',',
 'ĠInc',
 '.,',
 'Ġfor',
 'Ġa',
 'Ġseparate',
 'Ġlicense',
 'Ġfor',
 'Ġthis',
 'Ġsoftware',
 'Ġcode',
 ',',
 'Ġthe',
 'Ċ',
 '#',
 'Ġfollowing',
 'Ġterms',
 'Ġand',
 'Ġconditions',
 'Ġapply',
 ':',
 'Ċ',
 '#',
 'Ċ',
 '#',
 'ĠThis',
 'Ġprogram',
 'Ġis',
 'Ġfree',
 'Ġsoftware',
 ':',
 'Ġyou',
 'Ġcan',
 'Ġre',
 'distribute',
 'Ġit',
 'Ġand',
 '/',
 'or',
 'Ġmodify',
 'Ċ',
 '#',
 'Ġit',
 'Ġunder',
 'Ġthe',
 'Ġterms',
 'Ġof',
 'Ġthe',
 'ĠGNU',
 'ĠA',
 'ffer',
 'o',
 'ĠPublic',
 'ĠLicense',
 'Ġversion',
 'Ġ3',
 'Ġas',
 'Ċ',
 '#',
 'Ġpublished',
 'Ġby',
 'Ġthe',
 'ĠFree',
 'ĠSoftware',
 'ĠFound',
 'ation',
 '.',
 'Ċ',
 '#',
 'Ċ',
 '#',
 'ĠThis',
 'Ġprogra

We now have 16.7 million examples with 128 tokens each, which corresponds to about 2.1 billion tokens in total. For reference, OpenAI’s GPT-3 and Codex models are trained on 300 and 100 billion tokens, respectively, where the Codex models are initialized from the GPT-3 checkpoints. Our goal here is not to compete with these models, which can generate long, coherent texts, but to create a scaled-down version providing a quick autocomplete function for data scientists.

Now that we have the dataset ready, let’s set up the model!



### 3. Model

Our first step is to freshly initialize a GPT-2 model(not pretrained). We’ll use the same configuration for our model as for the small GPT-2 model, so we load the pretrained configuration, make sure that the tokenizer size matches the model vocabulary size and pass the `bos` and `eos` (beginning and end of sequence) token IDs.

In [36]:
# Check
from transformers import AutoTokenizer, AutoConfig

config = AutoConfig.from_pretrained("gpt2")
config, len(tokenizer)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

(GPT2Config {
   "activation_function": "gelu_new",
   "architectures": [
     "GPT2LMHeadModel"
   ],
   "attn_pdrop": 0.1,
   "bos_token_id": 50256,
   "embd_pdrop": 0.1,
   "eos_token_id": 50256,
   "initializer_range": 0.02,
   "layer_norm_epsilon": 1e-05,
   "model_type": "gpt2",
   "n_ctx": 1024,
   "n_embd": 768,
   "n_head": 12,
   "n_inner": null,
   "n_layer": 12,
   "n_positions": 1024,
   "reorder_and_upcast_attn": false,
   "resid_pdrop": 0.1,
   "scale_attn_by_inverse_layer_idx": false,
   "scale_attn_weights": true,
   "summary_activation": null,
   "summary_first_dropout": 0.1,
   "summary_proj_to_labels": true,
   "summary_type": "cls_index",
   "summary_use_proj": true,
   "task_specific_params": {
     "text-generation": {
       "do_sample": true,
       "max_length": 50
     }
   },
   "transformers_version": "4.53.0",
   "use_cache": true,
   "vocab_size": 50257
 },
 50000)

In [37]:
from transformers import AutoTokenizer, AutoConfig

config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=max_input_seq_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

config

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 128,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.53.0",
  "use_cache": true,
  "vocab_size": 50000
}

With that configuration, we can load a new model. Note that we don’t use the from_pretrained() function for our model, since we’re actually initializing a model ourself.

In [38]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel(config=config)

model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50000, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50000, bias=False)
)

In [41]:
model_size = sum(p.numel() for p in model.parameters())
print(f"GPT-2 size: {model_size / 1000**2:.1f}M parameters")

GPT-2 size: 124.2M parameters


### 4. Keywords Filtering

Since we are mainly interested in sensible autocompletion for the the data science libraries, it makes sense to give more weight to training samples that make more use of these libraries. We can easily identify these examples through the use of keywords such as plt, pd, sk, fit, and predict, which are the most frequent import names for matplotlib.pyplot, pandas, and sklearn as well as the fit/predict pattern of the latter. If these are each represented as a single token, we can easily check if they occur in the input sequence. Tokens can have a whitespace prefix, so we’ll also check for those versions in the tokenizer vocabulary. To verify that it works, we’ll add one test token which should be split into multiple tokens.

In [42]:
keywords = ["plt", "pd", "sk", "fit", "predict", "sns", "plot", " plt", " pd", " sk", " fit", " predict", " sns", " plot", "testtest"]

# Get the token ids for each keyword(we need to have sub tokenizer to deal with complex tokens which get split into subwords)
kw_token_ids = []
for kw in keywords:
    token_ids = tokenizer(kw, add_special_tokens=False).input_ids # list
    if len(token_ids) == 1:
        kw_token_ids.append(token_ids[0])
    else:
        print(f"'{kw}' is split into multiple tokens: {token_ids}")

'testtest' is split into multiple tokens: [1824, 1824]


In [43]:
kw_token_ids

[8436,
 4289,
 1201,
 2770,
 5431,
 23154,
 1389,
 2564,
 2604,
 2110,
 2872,
 4969,
 19207,
 1676]

In [44]:
tokenizer.convert_ids_to_tokens(kw_token_ids)

['plt',
 'pd',
 'sk',
 'fit',
 'predict',
 'sns',
 'plot',
 'Ġplt',
 'Ġpd',
 'Ġsk',
 'Ġfit',
 'Ġpredict',
 'Ġsns',
 'Ġplot']

### Weighted Loss Function

We can now write a custom loss function that takes inputs:
- input sequence
- logits
- key tokens we just selected

**We need to align the logits and inputs.**

- Prepare Labels from input sequence for LM models by shifting right
        
    The input sequence shifted by one to the right to form the labels, since the next token is the label for the current token. We can achieve this by starting the labels from the second token of the input sequence, since the model does not make a prediction for the first token `<bos>` anyway.

- Cut off last token logit
        
    Then we cut off the last logit, as we don’t have a ground truth label for the last token that follows the full input sequence.

This aligns them so that each predicted token has a corresponding ground-truth label.
   
With that we can compute the `loss per sample` and `count the occurrences of all keywords in each sample`. Finally, we calculate the `weighted average over all samples using the occurrences as weights`.

Since we don’t want to throw away all the samples that have no keywords, we add 1 to the weights.

We use this loss for training the model while for evaluation, we use the usual cross entropy loss from hf model.

### **Why do we shift inputs and labels?**

In causal (autoregressive) language modeling (like GPT), the model predicts token **t+1 given tokens up to t**. This means:

* Input: `[BOS] The cat sat on the`
* Labels: `The cat sat on the mat`

So the model learns to predict:

| Position | Input Token | Target (Label) |
| -------- | ----------- | -------------- |
| 0        | `[BOS]`     | `The`          |
| 1        | `The`       | `cat`          |
| 2        | `cat`       | `sat`          |
| 3        | `sat`       | `on`           |
| ...      | ...         | ...            |
| n-1      | `mat`       | ❌ (no target)  |

---

### So:  We **cut off the first label** to align things.

In code:

```python
shift_logits = logits[..., :-1, :]    # drop last time step prediction
shift_labels = inputs[..., 1:]        # drop first token, no label for it
```

Why?

* `logits[..., :-1, :]`: we don’t use the final prediction because there's no token to compare it to (nothing comes **after** the last token).
* `inputs[..., 1:]`: we shift the **labels** to the left — the model at position `t` should predict the token at `t+1`.

Note we **do use** the first input token (like BOS) — it predicts the first actual word.

We **don’t cut off the first input**, we cut off the **first label** to align properly.

---

| Step            | Reason                               |
| --------------- | ------------------------------------ |
| Cut last logit  | No token follows last input token    |
| Cut first label | First input (e.g., BOS) has no label |



In [93]:
import torch
from torch.nn import CrossEntropyLoss

def compute_weighted_loss(inputs, logits, kw_token_ids, alpha=1.0):
    labels = inputs[..., 1:].contiguous()  # shift right -  drop first token <bos>, no label for that  # (bs, seq_len + 1) -> (bs, seq_len)
    logits = logits[..., :-1, :].contiguous()  # drop last time step prediction - we dont have gt labels for <eos> token # (bs, seq_len + 1, vocab_size) -> (bs, seq_len, vocab_size)
    # Calculate per token loss - unreduced
    criterion = CrossEntropyLoss(reduction="none")
    loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))  # (bs*seq_len, vocab_size) (bs*seq_len) = (bs*seq_len)
    # Average loss per sample
    loss_per_sample = loss.view(logits.size(0), logits.size(1)).mean(axis=1)  # (bs, seq_len) -> # (bs,)
    # Weighting per sample
    weights = torch.stack([(inputs == kw).float() for kw in kw_token_ids]).sum(axis=[0, 2]) # (bs, seq_len + 1) -> list of (bs, seq_len + 1) kw_token_ids number of tensors -> (num_kws, bs, seq_len + 1) -> (bs,)
    weights = alpha * (1.0 + weights) # Scale the weights to make sure non matched tokens als0 get weight 1  # (bs,)
    weighted_loss = (loss_per_sample * weights).mean() # Get weighted average  (bs,) (bs,) -> scalar
    return weighted_loss

In [46]:
# Test for weights
import torch
test_inputs = torch.tensor([
    [1, 23, 56, 78, 90],   # sample 0
    [1, 90, 90, 10, 99],   # sample 1
])  # shape: (2, 5)

test_kw_token_ids = [90, 10]

mid_list = []
print(f"{test_inputs=}")
for kw in test_kw_token_ids:
    print(f"--------{kw=}---------")
    result = (test_inputs == kw).float()
    print(f"{result=}\n{result.shape=}")
    mid_list.append(result)

print(f"{mid_list=}")
stacked_mid_list = torch.stack(mid_list)
print(f"{stacked_mid_list.shape=}")
print(f"{stacked_mid_list.sum(axis=[0, 2])=}")
print(f"{stacked_mid_list.sum(axis=[0, 2]).shape=}")

test_inputs=tensor([[ 1, 23, 56, 78, 90],
        [ 1, 90, 90, 10, 99]])
--------kw=90---------
result=tensor([[0., 0., 0., 0., 1.],
        [0., 1., 1., 0., 0.]])
result.shape=torch.Size([2, 5])
--------kw=10---------
result=tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.]])
result.shape=torch.Size([2, 5])
mid_list=[tensor([[0., 0., 0., 0., 1.],
        [0., 1., 1., 0., 0.]]), tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.]])]
stacked_mid_list.shape=torch.Size([2, 2, 5])
stacked_mid_list.sum(axis=[0, 2])=tensor([1., 3.])
stacked_mid_list.sum(axis=[0, 2]).shape=torch.Size([2])


### DataLoaders

In [49]:
import torch
from torch.utils.data import DataLoader

tokenized_ds.set_format("torch")
train_dl = DataLoader(tokenized_ds["train"], batch_size=32, shuffle=True)
eval_dl = DataLoader(tokenized_ds["validation"], batch_size=32)

In [51]:
for batch in train_dl: # for test
    break

batch.keys()

dict_keys(['input_ids'])

In [52]:
batch["input_ids"].shape

torch.Size([32, 128])

In [58]:
outputs = model(batch["input_ids"])
outputs.keys()

odict_keys(['logits', 'past_key_values'])

`past_key_values` refers to a mechanism for caching and reusing previously computed key and value states within the self-attention layers during sequential processing, such as text generation. This caching mechanism significantly speeds up the process by avoiding redundant calculations for parts of the input that haven't changed

In Hugging Face's implementation of GPT-2, the causal mask is automatically generated during the forward pass of the model when performing causal language modeling (CLM), even if an attention_mask isn't explicitly provided. This is because GPT-2 is inherently designed for CLM, where each token can only attend to previous tokens in the sequence.

Here's how it works:
1. Causal Language Modeling:
GPT-2 is trained to predict the next token in a sequence, given the preceding tokens. This autoregressive nature means that during training and inference, the model needs to attend only to the past tokens and not to future tokens.
2. Automatic Mask Creation:
The transformers library, when processing input for GPT-2 with a CLM objective, automatically creates a causal mask. This mask is a lower triangular matrix where the upper triangle is filled with negative infinity (or a very large negative number) and the diagonal and lower triangle contain zeros. This effectively prevents the model from attending to future tokens.
3. Application to Attention Weights:
The generated causal mask is then applied to the attention weights during the scaled_dot_product_attention operation within the attention mechanism. This ensures that each token in the sequence attends only to the tokens that precede it in the sequence.
4. No Need for Explicit Mask:
Because the causal mask is automatically generated based on the CLM objective, users don't need to provide an attention_mask when using GPT-2 for tasks like text generation or language modeling. The library handles the masking internally

In [59]:
outputs["logits"].shape # samples in batch, tokens, vocabs

torch.Size([32, 128, 50000])

In [60]:
print(outputs.logits[0, -1].topk(5))  # top 5 logits for last token in 1st sample

torch.return_types.topk(
values=tensor([2.2307, 2.1719, 2.0892, 2.0749, 2.0101], grad_fn=<TopkBackward0>),
indices=tensor([ 3793, 32001, 45219, 38868,  5831]))


### Optimizer with grouped params for weight decay

Next, we group the parameters so that the optimizer knows which ones will get an additional weight decay. Usually, all bias and LayerNorm weights terms are exempt from this,

In [63]:
weight_decay = 0.1

def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for name, param in model.named_parameters():
        if any(nd in name for nd in no_decay):
            params_without_wd.append(param)
        else:
            params_with_wd.append(param)
    return [
        {"params": params_with_wd, "weight_decay": weight_decay},
        {"params": params_without_wd, "weight_decay": 0.0}
    ]

In [64]:
from torch.optim import AdamW

optimizer = AdamW(get_grouped_params(model), lr=5e-4)

### Accelerator

In [65]:
import accelerate
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision="fp16")
model, optimizer, train_dl, eval_dl = accelerator.prepare(model, optimizer, train_dl, eval_dl)

### LR Scheduler

In [66]:
from transformers import get_scheduler

num_epochs = 1
num_steps_per_epoch = len(train_dl)
num_training_steps = num_epochs * num_steps_per_epoch
scheduler = get_scheduler("linear", optimizer=optimizer, num_training_steps=num_training_steps, num_warmup_steps=1_000)

### Evaluation Step

Since we want to evaluate the model regularly on the validation set during training, let’s write a function for that as well. It just runs through the evaluation dataloader and gathers all the losses across processes:

In [83]:
len(eval_dl), len(train_dl)

(426, 42986)

In [84]:
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dl):
        with torch.no_grad():
            outputs = model(batch["input_ids"], labels=batch["input_ids"])  # dict_keys(['loss', 'logits', 'past_key_values'])
            outputs.loss = outputs.loss.reshape(1) # Usual CE loss # tensor(10.9658) of torch.Size([]) -> torch.Size([1])
        losses.append(accelerator.gather(outputs.loss))

    loss = torch.mean(torch.cat(losses)) # [len(val_dataloader) of torch.Size([1])] => torch.Size([426]) -> scalar
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = float("inf")
    return loss.item(), perplexity.item()

In [85]:
# Before training, run a quick test to see if the evaluation function works properly
evaluate()

(10.972529411315918, 58251.75)

Those are very high values for loss and perplexity, but that’s not surprising as we haven’t trained the model yet.

### Saving to Hub

In [89]:
from huggingface_hub import HfApi, create_repo, get_full_repo_name, upload_folder
from google.colab import userdata
import os

api = HfApi(token=userdata.get("HF_TOKEN"))
repo_name = "codeparrot-code-generator-gpt2"
repo_id = get_full_repo_name(repo_name, token=api.token)
print(repo_id)

output_dir = "codeparrot-code-generator-gpt2"
os.makedirs(output_dir, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
try:
    create_repo(repo_id=repo_id, repo_type="model", private=False, token=api.token, exist_ok=True)
    print("Repo created")
except Exception as e:
    print(f"Error creating repo: {e}")

BhavanaMalla/codeparrot-code-generator-gpt2
Repo created


### Training

In the training loop we iterate over the dataloader and pass the batches to the model. With the logits, we can then evaluate our custom loss function. We scale the loss by the number of gradient accumulation steps so as not to create larger losses when aggregating more steps. Before we optimize, we also clip the gradients for better convergence. Finally, every few steps we evaluate the model on the evaluation set with our new evaluate() function.

In [96]:
from tqdm.notebook import tqdm

gradient_accumulation_steps = 8
eval_steps = 5_000

model.train()
completed_steps = 0
for epoch in range(num_epochs):
    for step, batch in tqdm(enumerate(train_dl, start=1), total=num_training_steps):
        logits = model(batch["input_ids"]).logits
        loss = compute_weighted_loss(batch["input_ids"], logits, kw_token_ids)
        if step % 100 == 0:
            accelerator.print({
                "steps": completed_steps,
                "loss/train": loss.item() * gradient_accumulation_steps,
            })
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        if step % gradient_accumulation_steps == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1
        if (step % (eval_steps * gradient_accumulation_steps)) == 0:
            eval_loss, perplexity = evaluate()
            accelerator.print({
                "loss/eval": eval_loss,
                "perplexity": perplexity
            })
            model.train() # put back to training mode
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
            if accelerator.is_main_process:
                tokenizer.save_pretrained(output_dir)
                try:
                    upload_folder(repo_id=repo_id, folder_path=output_dir, repo_type="model", commit_message=f"Training in progress epoch {epoch}", token=api.token)
                except Exception as e:
                    print(f"Failed to upload files: {e}")



  0%|          | 0/42986 [00:00<?, ?it/s]

{'steps': 12, 'loss/train': 125.8653793334961}
{'steps': 24, 'loss/train': 116.67231750488281}
{'steps': 37, 'loss/train': 101.1414566040039}
{'steps': 49, 'loss/train': 101.03474426269531}
{'steps': 62, 'loss/train': 94.24048614501953}
{'steps': 74, 'loss/train': 90.16767120361328}
{'steps': 87, 'loss/train': 92.46833801269531}
{'steps': 99, 'loss/train': 80.85487365722656}
{'steps': 112, 'loss/train': 61.24872970581055}
{'steps': 124, 'loss/train': 80.74494934082031}
{'steps': 137, 'loss/train': 89.08897399902344}
{'steps': 149, 'loss/train': 72.76786041259766}
{'steps': 162, 'loss/train': 79.36128997802734}
{'steps': 174, 'loss/train': 70.40917205810547}
{'steps': 187, 'loss/train': 69.1443862915039}
{'steps': 199, 'loss/train': 75.91541290283203}
{'steps': 212, 'loss/train': 63.207740783691406}
{'steps': 224, 'loss/train': 71.40511322021484}
{'steps': 237, 'loss/train': 57.26231002807617}
{'steps': 249, 'loss/train': 67.12154388427734}
{'steps': 262, 'loss/train': 51.47281646728515

model.safetensors:   0%|          | 0.00/497M [00:00<?, ?B/s]

{'steps': 5012, 'loss/train': 29.34549903869629}
{'steps': 5024, 'loss/train': 35.6705207824707}
{'steps': 5037, 'loss/train': 22.78495979309082}
{'steps': 5049, 'loss/train': 25.182899475097656}
{'steps': 5062, 'loss/train': 23.312671661376953}
{'steps': 5074, 'loss/train': 24.776451110839844}
{'steps': 5087, 'loss/train': 29.51312255859375}
{'steps': 5099, 'loss/train': 21.889625549316406}
{'steps': 5112, 'loss/train': 30.01121711730957}
{'steps': 5124, 'loss/train': 25.56147003173828}
{'steps': 5137, 'loss/train': 19.027311325073242}
{'steps': 5149, 'loss/train': 25.947372436523438}
{'steps': 5162, 'loss/train': 17.84716796875}
{'steps': 5174, 'loss/train': 18.3210391998291}
{'steps': 5187, 'loss/train': 22.302688598632812}
{'steps': 5199, 'loss/train': 27.088685989379883}
{'steps': 5212, 'loss/train': 24.18901824951172}
{'steps': 5224, 'loss/train': 19.475290298461914}
{'steps': 5237, 'loss/train': 16.511463165283203}
{'steps': 5249, 'loss/train': 28.82818603515625}
{'steps': 5262,


#### 1. Why divide the loss by `gradient_accumulation_steps`?

```python
loss = loss / gradient_accumulation_steps
```

#### Gradient accumulation recap:

If your model or batch is too large to fit in memory, you can **simulate a larger batch size** by accumulating gradients over multiple forward-backward passes before doing an optimizer step.

So if:

* `gradient_accumulation_steps = 8`
* You're using a batch size of 16
* Then you're simulating a batch size of `16 * 8 = 128`

#### Why divide loss?

Loss **should be averaged per sample** across the full batch before backpropagation. Since you're splitting your big batch into `gradient_accumulation_steps` smaller steps, you also need to **scale the loss down proportionally** to avoid over-updating the gradients.

So we divide by `gradient_accumulation_steps` to keep the effective learning rate correct.

If you **don’t** divide the loss, you'd effectively multiply the gradient magnitude by 8 every time.

---

#### 2. Why multiply it again when printing?

```python
if step % 100 == 0:
    accelerator.print({
        "steps": completed_steps,
        "loss/train": loss.item() * gradient_accumulation_steps,
    })
```

Because **you divided the loss earlier**, what you're printing is now the *per-microbatch* loss. That's usually much smaller than you'd expect.

So multiplying by `gradient_accumulation_steps` restores the original loss scale for logging, making it easier to interpret and compare across runs.

**In short:**

* `loss`: used for training (needs to be scaled down)
* `loss * grad_accum_steps`: used for logging (restore full batch scale)

---

#### Eval step condition

```python
if (step % (eval_steps * gradient_accumulation_steps)) == 0:
```

#### Purpose:

You want to evaluate the model **after a fixed number of optimizer updates**, **not forward passes**.

Since:

* `eval_steps = 5000` → you want to eval every 5000 gradient updates
* But you're doing one optimizer step every `gradient_accumulation_steps` forward passes

Therefore:

* `step % (eval_steps * gradient_accumulation_steps) == 0`
* Means: "Have we done `eval_steps` optimizer steps?"

So you evaluate every **5000 updates**, even if gradient accumulation is active.

---


| Line                                                          | Purpose                                                                          |
| ------------------------------------------------------------- | -------------------------------------------------------------------------------- |
| `loss = loss / gradient_accumulation_steps`                   | Prevents over-scaling gradients during accumulation                              |
| `loss.item() * gradient_accumulation_steps`                   | Logs the original-scale loss for visibility                                      |
| `if (step % (eval_steps * gradient_accumulation_steps)) == 0` | Triggers eval after the right number of *optimizer updates* (not forward passes) |



### Inference

In [116]:
from transformers import pipeline

code_generator = pipeline("text-generation", model="BhavanaMalla/codeparrot-code-generator-gpt2",
                          max_length=1000, pad_token_id=0, eos_token_id=0)

Device set to use cuda:0


In [117]:
text = """\
# Create some data
x = np.random.rand(100)
y = np.random.rand(100)

# create scatter plot with x, y

"""

print(code_generator(text, num_return_sequences=1)[0]["generated_text"])


# Create some data
x = np.random.rand(100)
y = np.random.rand(100)

# create scatter plot with x, y


fig = plt.figure(figsize=(10, 5))


In [124]:
text = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
"""
print(code_generator(text, num_return_sequences=1)[0]["generated_text"])

# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
x_1 = pd.DataFrame(x, columns=['x',


In [131]:
text = """\
# dataframe with profession, income and name
df = pd.DataFrame({'profession': x, 'income':y, 'name': z})

# calculate the mean income per profession
"""
print(code_generator(text, num_return_sequences=1)[0]["generated_text"])

# dataframe with profession, income and name
df = pd.DataFrame({'profession': x, 'income':y, 'name': z})

# calculate the mean income per profession
df['income'].index


In [135]:
text = """\
# import random forest regressor from scikit-learn
from sklearn.ensemble import RandomForestRegressor

# fit random forest model with 300 estimators on X, y
"""
print(code_generator(text, num_return_sequences=1)[0]["generated_text"])

# import random forest regressor from scikit-learn
from sklearn.ensemble import RandomForestRegressor

# fit random forest model with 300 estimators on X, y
random_forest = RandomForestRegressor(max_depth=2, random_state


### Final Version

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from collections import defaultdict

from transformers import AutoTokenizer, AutoConfig, GPT2LMHeadModel, get_scheduler, pipeline

import accelerate
from accelerate import Accelerator

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from tqdm.notebook import tqdm

from huggingface_hub import HfApi, create_repo, upload_folder, get_full_repo_name
from google.colab import userdata
import os

# Prepare Dataset from scratch
# ds = load_dataset("transformersbook/codeparrot-train", split="train", streaming=True)

# filters = ["pandas", "numpy", "matplotlib", "torch", "transformers", "seaborn", "sklearn"]

# def check_for_keywords(text, filters):
#     return any(kw.lower() in text.lower() for kw in filters)

# # Preprocess
# def filter_streaming_dataset(dataset, filters):
#     filtered_samples = defaultdict(list)
#     for sample in iter(dataset):
#         if check_for_keywords(sample, filters):
#             for k, v in sample.items():
#                 filtered_samples[k].append(v)
#     return Dataset.from_dict(filtered_samples)

# filtered_ds = filter_streaming_dataset(ds, filters)

# Load already prepared dataset
train_ds = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
validation_ds = load_dataset("huggingface-course/codeparrot-ds-valid", split="validation")

raw_ds = DatasetDict({
    "train": train_ds.shuffle(seed=42).select(range(10000)),
    "validation": validation_ds.shuffle(seed=42).select(range(500))
})

# Preprocess
checkpoint = "huggingface-course/code-search-net-tokenizer"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
max_input_seq_length = 128

def preprocess(examples):
    input_ids = []
    for text in examples["content"]:
        inputs = tokenizer(text, max_length=max_input_seq_length, truncation=True, return_overflowing_tokens=True, return_length=True, stride=50)
        for input, length in zip(inputs["input_ids"], inputs["length"]):
            if length == max_input_seq_length:
                input_ids.append(input)
    return {"input_ids": input_ids}

tokenized_ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds["train"].column_names)
tokenized_ds.set_format("torch")

# Model
config = AutoConfig.from_pretrained("gpt2",
                                    n_ctx=max_input_seq_length,
                                    vocab=len(tokenizer),
                                    eos_token_id=tokenizer.eos_token_id,
                                    bos_token_id=tokenizer.bos_token_id)

model = GPT2LMHeadModel(config)
print(f"Our Model GPT2 has {sum([p.numel() for p in model.parameters()]) / (1000**2):.1f}M params")

# Loss
key_tokens = ["plt", "pd", "sk", "fit", "predict", "sns", "plot", " plt", " pd", " sk", " fit", " predict", " sns", " plot", "testtest"]
key_token_ids = []
for kw in key_tokens:
    ids = tokenizer(kw, add_special_tokens=False)["input_ids"]
    if len(ids) == 1:
        key_token_ids.append(ids[0])

def compute_weighted_loss(inputs, logits, key_token_ids, alpha=1.0):
    labels = inputs[..., 1:].contiguous()  # (bs, seq_len + 1) -> (bs, seq_len)
    logits = logits[..., :-1, :]. contiguous() # (bs, seq_len + 1, voc_size) -> (bs, seq_len, voc_size)
    criterion = CrossEntropyLoss(reduction="none")
    loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))  # (bs*seq_len, voc_size) (bs*seq_len) => (bs*seq_len)
    loss = loss.view(logits.size(0), logits.size(1)).mean(axis=1) # loss per sample (bs*seq_len) => (bs, seq_len) => (bs,)
    weights = torch.stack([(kw == inputs).float() for kw in key_token_ids]).sum(axis=[0, 2]) # (len(key_token_ids), bs, seq_len)
    weights = alpha * (1 + weights)
    weighted_loss = (loss * weights).mean()
    return weighted_loss

# Dataloaders
train_dl = DataLoader(tokenized_ds["train"], shuffle=True, batch_size=32)
eval_dl = DataLoader(tokenized_ds["validation"], batch_size=32)

# Optimizer
weight_decay = 1.0
def grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_wo_decay = []
    params_with_decay = []
    for name, param in model.named_parameters():
        if any(nd in name for nd in no_decay):
            params_wo_decay.append(param)
        else:
            params_with_decay.append(param)
        return [{"params": params_wo_decay, "weight_decay": 0.0},
                {"params": params_with_decay, "weight_decay": weight_decay}] # list of dicts

optimizer = AdamW(grouped_params(model), lr=5e-4)

# Accelerator
accelerator = Accelerator(mixed_precision="fp16")
model, optimizer, train_dl, eval_dl = accelerator.prepare(model, optimizer, train_dl, eval_dl)

# Scheduler
num_epochs = 1
num_steps_per_epoch = len(train_dl)
num_training_steps = num_epochs * num_steps_per_epoch
scheduler = get_scheduler("linear", optimizer=optimizer, num_training_steps=num_training_steps, num_warmup_steps=1_000)

# Evaluation
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dl):
        with torch.no_grad():
            outputs = model(batch["input_ids"], labels=batch["input_ids"])
        outputs.loss = outputs.loss.reshape(1)
        losses.append(accelerator.gather(outputs.loss))
    loss = torch.mean(torch.cat(losses))
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = float("inf")
    return loss.item(), perplexity.item()

# Save to Hf
api = HfApi(token=userdata.get("HF_TOKEN"))
repo_name = "codeparrot-code-generator-gpt2-self"
repo_id = get_full_repo_name(repo_name, token=api.token)
print(repo_id)

output_dir = "codeparrot-code-generator-gpt2-self"
os.makedirs(output_dir, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
try:
    create_repo(repo_id=repo_id, repo_type="model", private=False, token=api.token, exist_ok=True)
    print("Repo created")
except Exception as e:
    print(f"Error creating repo: {e}")

# Training
gradient_accumulation_steps = 8
eval_steps = 1_000
model.train()
completed_steps = 0
for epoch in range(num_epochs):
    for step, batch in tqdm(enumerate(train_dl, start=1), total=num_training_steps):
        outputs = model(batch["input_ids"])
        loss = compute_weighted_loss(batch["input_ids"], outputs.logits, key_token_ids)
        if step % 100 == 0:
            accelerator.print({
                "steps": completed_steps,
                "loss/train": loss.item() * gradient_accumulation_steps
            })
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss) # backward here
        if (step % gradient_accumulation_steps) == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()  # update params here after accumulating for gradient_accumulation_steps
            scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1
        if (step % (eval_steps * gradient_accumulation_steps)) == 0:
            val_loss, perplexity = evaluate()
            accelerator.print({
                "loss/val": val_loss,
                "perplexity": perplexity
            })
            model.train() # put back to training mode
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
            if accelerator.is_main_process:
                tokenizer.save_pretrained(output_dir)
                try:
                    upload_folder(repo_id=repo_id, repo_type="model", commit_message=f"Training in progress epoch {epoch}",
                                  folder_path=output_dir, token=api.token)
                except Exception as e:
                    print(f"Failed to Save checkpoint to Hugging face: {e}")

# Inference
generator = pipeline("text-generation", model="BhavanaMalla/codeparrot-code-generator-gpt2-self",
                     max_length=1000, pad_token_id=0, eos_token_id=0)
text = """
# Create a function to add 2 numbers and return result
def add_numbers(x, y):

"""
generator(text, num_return_sequences=1)[0]["generated_text"]

In [136]:
import gradio as gr

# Set up the Gradio interface
def generate_code(text):
    try:
        result = code_generator(text, num_return_sequences=1)[0]["generated_text"]
        return result
    except Exception as e:
        return f"Error generating code: {e}"

demo = gr.Interface(
    fn=generate_code,
    inputs=gr.Textbox(label="Enter Code to be completed.", lines=6, placeholder="Type or paste a product code here..."),
    outputs=gr.Textbox(label="Generated Code"),
    title="Code Generator with GPT2",
    description="This demo uses a GPT2 model trained from scratch on Code parrot dataset to generate python code. Trained on Data science libraries python code snippets."
)
# Launch the demo
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e077047f8b9541f7fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
